# Revision diagnostic: 05_lora_diag

Runs the LoRA training diagnostic (`scripts/lora_diag.py`): TinyLlama, 200 steps per arm, fp16 AMP against fp32 adapters and three one-change variants, every step logged.

Launched with `scripts/kaggle_run.sh 05_lora_diag <commit>`, which sets `REPO_COMMIT` below. Cells: clone the
repository at that commit, install, check the GPU, run `scripts/lora_diag.py` in one process on GPU 0, print its
log. Everything under `results/revision/lora_diag/` is zipped into `/kaggle/working/`; the log is in `/kaggle/working/logs/`.

In [ ]:
REPO_COMMIT = "SET_BY_KAGGLE_RUN"  # written by scripts/kaggle_run.sh
REPO_URL = "https://github.com/Thommartial/llm-unlearning-experiments.git"
GPU_SEEDS = {0: None}  # one process on GPU 0

In [ ]:
import os, subprocess, sys
assert REPO_COMMIT != "SET_BY_KAGGLE_RUN", "launch with scripts/kaggle_run.sh, which sets REPO_COMMIT"
# fewer fragmented reserves; set before torch starts CUDA here or in any subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
REPO = "/tmp/repo"  # outside /kaggle/working: only results and logs are kept as output
subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(["git", "checkout", "-q", REPO_COMMIT], check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
# sys.path.insert below reaches this kernel only: every subprocess gets src/ on PYTHONPATH as well
ENV = {**os.environ, "PYTHONPATH": os.pathsep.join(filter(None, [f"{REPO}/src", os.environ.get("PYTHONPATH")]))}

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "peft",
                'transformers==5.9.0', 'peft==0.19.1', 'datasets==4.8.5', 'accelerate==1.13.0'], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)  # the package itself
# Kaggle ships torchao 0.10, which peft rejects when it builds LoRA layers; nothing here uses torchao
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
versions = subprocess.run([sys.executable, "-c", "import importlib.metadata as m; from peft.import_utils import is_torchao_available; print({p: m.version(p) for p in ('torch', 'transformers', 'peft')}); print('torchao seen by peft:', is_torchao_available()); assert not is_torchao_available()"], capture_output=True, text=True, env=ENV)
print(versions.stdout, versions.stderr)
assert versions.returncode == 0, "the version check failed (is torchao still visible to peft?)" 

In [ ]:
sys.path.insert(0, "src")
import torch, transformers, peft
n_gpu = torch.cuda.device_count()
print("torch", torch.__version__, "CUDA", torch.version.cuda, "transformers", transformers.__version__, "peft", peft.__version__)
for i in range(n_gpu):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
assert n_gpu >= len(GPU_SEEDS), f"need {len(GPU_SEEDS)} GPU(s), found {n_gpu}" 

In [ ]:
os.makedirs("/kaggle/working/logs", exist_ok=True)
log = "/kaggle/working/logs/05_lora_diag.log"
cmd = [sys.executable, "scripts/lora_diag.py", "--steps", "200", "--device", "cuda",
       "--out", "results/revision/lora_diag"]
with open(log, "w") as f:
    run = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT,
                         env={**ENV, "CUDA_VISIBLE_DEVICES": "0", "PYTHONUNBUFFERED": "1"})
print(open(log).read())

In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/05_lora_diag", "zip", "results/revision/lora_diag")
print("results ->", archive)
assert run.returncode == 0, "the diagnostic failed: see the log above" 